# A3-M1 · 01 토큰 나누기 기초
글자·띄어쓰기·형태소(kiwi)·서브워드(BPE)로 같은 문장을 잘라 보고, BPE 병합 과정을 손으로 따라갑니다.

## 01-1. 준비 — 설치 확인과 예문

In [3]:
# 터미널: pip install kiwipiepy tokenizers transformers tiktoken scikit-learn
from kiwipiepy import Kiwi
import tokenizers, transformers, tiktoken
print("kiwipiepy 준비 완료 | tokenizers", tokenizers.__version__, "| transformers", transformers.__version__)

sentence = "택배비 얼마예요? 35kg 냉장고(RF85T)도 free shipping 되나요"
print(sentence)

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


kiwipiepy 준비 완료 | tokenizers 0.23.3 | transformers 5.21.0
택배비 얼마예요? 35kg 냉장고(RF85T)도 free shipping 되나요


## 01-2. 같은 문장, 다섯 가지 자르기

In [4]:
import re
kiwi = Kiwi()
enc = tiktoken.get_encoding("o200k_base")

ways = {
    "글자":      list(sentence.replace(" ", "")),
    "띄어쓰기":  sentence.split(),
    "정규식 단어": re.findall(r"\w+|[^\w\s]", sentence),
    "형태소(kiwi)": [t.form for t in kiwi.tokenize(sentence)],
    "BPE(tiktoken o200k)": [enc.decode([i]) for i in enc.encode(sentence)],
}
for name, toks in ways.items():
    print(f"{name:<18} {len(toks):>2}개  {toks}")

글자                 38개  ['택', '배', '비', '얼', '마', '예', '요', '?', '3', '5', 'k', 'g', '냉', '장', '고', '(', 'R', 'F', '8', '5', 'T', ')', '도', 'f', 'r', 'e', 'e', 's', 'h', 'i', 'p', 'p', 'i', 'n', 'g', '되', '나', '요']
띄어쓰기                7개  ['택배비', '얼마예요?', '35kg', '냉장고(RF85T)도', 'free', 'shipping', '되나요']
정규식 단어             12개  ['택배비', '얼마예요', '?', '35kg', '냉장고', '(', 'RF85T', ')', '도', 'free', 'shipping', '되나요']
형태소(kiwi)          18개  ['택배비', '얼마', '이', '예요', '?', '35', 'kg', '냉장고', '(', 'RF', '85', 'T', ')', '도', 'free', 'shipping', '되', '나요']
BPE(tiktoken o200k) 24개  ['택', '배', '비', ' 얼마', '예', '요', '?', ' ', '35', 'kg', ' �', '�', '장', '고', '(R', 'F', '85', 'T', ')', '도', ' free', ' shipping', ' 되', '나요']


## 02-1. kiwi 형태소 분석 — 결과 객체 읽기

In [5]:
tokens = kiwi.tokenize("택배비 얼마예요?? 35키로 냉장고는요")
for t in tokens:
    print(f"{t.form:<6} {t.tag:<4} 시작 {t.start:>2} 길이 {t.len}")
print("첫 토큰 통째로:", tokens[0])

택배비    NNG  시작  0 길이 3
얼마     NNG  시작  4 길이 2
이      VCP  시작  6 길이 0
예요     EF   시작  6 길이 2
??     SF   시작  8 길이 2
35     SN   시작 11 길이 2
키로     NNB  시작 13 길이 2
냉장고    NNG  시작 16 길이 3
는      JX   시작 19 길이 1
요      MM   시작 20 길이 1
첫 토큰 통째로: Token(form='택배비', tag='NNG', start=0, len=3)


## 02-3. 필요한 품사만 골라 쓰기 — 검색용 토크나이저 만들기

In [6]:
from kiwipiepy.utils import Stopwords

KEEP = {"NNG", "NNP", "NNB", "SL", "SN", "VV", "VA", "XR"}      # 일반·고유·의존명사, 외국어, 숫자, 동사, 형용사, 어근
def kiwi_tokens(text, keep=KEEP):
    return [t.form for t in kiwi.tokenize(text) if t.tag in keep]

q = "냉장고는 언제쯤 오나요?"
print("전체      :", [(t.form, t.tag) for t in kiwi.tokenize(q)])
print("품사 골라냄:", kiwi_tokens(q))
print("불용어 제거:", [t.form for t in kiwi.tokenize(q, stopwords=Stopwords())])

전체      : [('냉장고', 'NNG'), ('는', 'JX'), ('언제', 'NP'), ('쯤', 'XSN'), ('오', 'VV'), ('나요', 'EF'), ('?', 'SF')]
품사 골라냄: ['냉장고', '오']
불용어 제거: ['냉장고', '언제', '쯤', '오', '나요']


## 02-4. 사용자 사전 · 띄어쓰기 교정 · 문장 나누기

In [7]:
text = "CNC 머시닝센터 3호기에서 불량 발생"
print("사전 추가 전:", [(t.form, t.tag) for t in kiwi.tokenize(text)])
kiwi.add_user_word("머시닝센터", "NNP")                     # 고유명사로 등록
kiwi.add_user_word("3호기", "NNP")
print("사전 추가 후:", [(t.form, t.tag) for t in kiwi.tokenize(text)])

print("띄어쓰기 교정:", kiwi.space("배송비얼마인가요냉장고도무료배송인가요"))
print("문장 나누기 :", [s.text for s in kiwi.split_into_sents("배송 언제 와요? 어제 주문했어요. 빨리 받고 싶어요")])

사전 추가 전: [('CNC', 'SL'), ('머시닝센터', 'NNP'), ('3', 'SN'), ('호기', 'NNB'), ('에서', 'JKB'), ('불량', 'NNG'), ('발생', 'NNG')]
사전 추가 후: [('CNC', 'SL'), ('머시닝센터', 'NNP'), ('3호기', 'NNP'), ('에서', 'JKB'), ('불량', 'NNG'), ('발생', 'NNG')]
띄어쓰기 교정: 배송비 얼마인가요 냉장고도 무료 배송인가요
문장 나누기 : ['배송 언제 와요?', '어제 주문했어요.', '빨리 받고 싶어요']


## 03-1. BPE 병합을 손으로 따라가기

In [8]:
from collections import Counter

words = {"배송": 6, "배송비": 4, "배송지": 3, "배달": 2, "배달비": 2}     # 단어: 등장 횟수
splits = {w: list(w) for w in words}                                      # 처음엔 글자 단위
vocab = sorted({ch for w in words for ch in w})
print("시작 어휘:", vocab)

for step in range(1, 5):
    pairs = Counter()
    for w, freq in words.items():                                         # 이웃한 두 조각 쌍 세기
        s = splits[w]
        for a, b in zip(s, s[1:]):
            pairs[(a, b)] += freq
    (a, b), cnt = pairs.most_common(1)[0]                                 # 가장 많이 나온 쌍
    for w in words:                                                       # 그 쌍을 하나로 합친다
        s, out, i = splits[w], [], 0
        while i < len(s):
            if i < len(s) - 1 and s[i] == a and s[i + 1] == b:
                out.append(a + b); i += 2
            else:
                out.append(s[i]); i += 1
        splits[w] = out
    vocab.append(a + b)
    print(f"병합 {step}: '{a}'+'{b}' (함께 {cnt}번) → {splits}")
print("최종 어휘:", vocab)

시작 어휘: ['달', '배', '비', '송', '지']
병합 1: '배'+'송' (함께 13번) → {'배송': ['배송'], '배송비': ['배송', '비'], '배송지': ['배송', '지'], '배달': ['배', '달'], '배달비': ['배', '달', '비']}
병합 2: '배송'+'비' (함께 4번) → {'배송': ['배송'], '배송비': ['배송비'], '배송지': ['배송', '지'], '배달': ['배', '달'], '배달비': ['배', '달', '비']}
병합 3: '배'+'달' (함께 4번) → {'배송': ['배송'], '배송비': ['배송비'], '배송지': ['배송', '지'], '배달': ['배달'], '배달비': ['배달', '비']}
병합 4: '배송'+'지' (함께 3번) → {'배송': ['배송'], '배송비': ['배송비'], '배송지': ['배송지'], '배달': ['배달'], '배달비': ['배달', '비']}
최종 어휘: ['달', '배', '비', '송', '지', '배송', '배송비', '배달', '배송지']
